# Combining LCEL Chains

## Setup

#### After you download the code from the github repository in your computer
In terminal:
* cd project_name
* pyenv local 3.11.4
* poetry install
* poetry shell

#### To open the notebook with Jupyter Notebooks
In terminal:
* jupyter lab

Go to the folder of notebooks and open the right notebook.

#### To see the code in Virtual Studio Code or your editor of choice.
* open Virtual Studio Code or your editor of choice.
* open the project-folder
* open the 007-combining-chain.py file

## Create your .env file
* In the github repo we have included a file named .env.example
* Rename that file to .env file and here is where you will add your confidential api keys. Remember to include:
* OPENAI_API_KEY=your_openai_api_key
* LANGCHAIN_TRACING_V2=true
* LANGCHAIN_ENDPOINT=https://api.smith.langchain.com
* LANGCHAIN_API_KEY=your_langchain_api_key
* LANGCHAIN_PROJECT=your_project_name

We will call our LangSmith project **007-combining-chain**.

## Connect with the .env file located in the same directory of this notebook

If you are using the pre-loaded poetry shell, you do not need to install the following package because it is already pre-loaded for you:

In [ ]:
#!pip install python-dotenv

In [5]:
import os
from dotenv import load_dotenv, find_dotenv
_ = load_dotenv(find_dotenv())
openai_api_key = os.environ["OPENAI_API_KEY"]

#### Install LangChain

If you are using the pre-loaded poetry shell, you do not need to install the following package because it is already pre-loaded for you:

In [ ]:
#!pip install langchain

In [3]:
#!pip install langchain lanchain-community

## Connect with an LLM and start a conversation with it

If you are using the pre-loaded poetry shell, you do not need to install the following package because it is already pre-loaded for you:

In [5]:
#!pip install langchain-openai

* For this project, we will use a current OpenAI chat model (gpt-6-luna, defined in MODEL)

#### Update note: current OpenAI models
* The original example is very good for learning the concept, and it worked
  with the OpenAI models available when the course was recorded: ChatOpenAI
  used gpt-3.5-turbo.
* OpenAI has since retired that model. With the current models, we would do
  it like this: the same ChatOpenAI chat model, with gpt-6-luna. The model
  name is defined once in MODEL and reused in the rest of the notebook,
  including the example that used ChatOpenAI() without a model name (its
  default model was also gpt-3.5-turbo).
* This project's langchain-openai sends temperature=0.7 by default; recent
  OpenAI models only accept 1, so we set temperature=1 explicitly.
* Check available models at developers.openai.com/api/docs/deprecations.

In [ ]:
from langchain_openai import ChatOpenAI

# Original version: this example used gpt-3.5-turbo, which was a good
# choice when the course was recorded. OpenAI has since retired it.
# Today we use a current chat model: small, fast and affordable for practice.
# It is defined once here and reused in the rest of the notebook.
MODEL = "gpt-6-luna"

# This version of langchain-openai sends temperature=0.7 by default,
# but recent OpenAI models only accept the default value (1).
# That is why every model in this notebook sets temperature=1 explicitly.
model = ChatOpenAI(model=MODEL, temperature=1)

## Coercion: a chain inside another chain
* Remember: almost any component in LangChain (prompts, models, output parsers, etc) can be used as a Runnable.
* **Runnables can be chained together using the pipe operator `|`. The resulting chains of runnables are also runnables themselves**.

In [ ]:
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate.from_template("tell me a sentence about {politician}")

# Original version: gpt-3.5-turbo, now retired. We reuse MODEL.
model = ChatOpenAI(model=MODEL, temperature=1)

chain = prompt | model | StrOutputParser()

In [ ]:
chain.invoke("Chamberlain")

#### Coercion: combine a chain with other Runnables to create a new chain.
* See how in the `composed_chain` we are including the previous `chain`:

In [20]:
from langchain_core.output_parsers import StrOutputParser

historian_prompt = ChatPromptTemplate.from_template("Was {politician} positive for Humanity?")

composed_chain = {"politician": chain} | historian_prompt | model | StrOutputParser()

In [ ]:
composed_chain.invoke({"politician": "Lincoln"})

In [ ]:
composed_chain.invoke({"politician": "Attila"})

## Another example: a chain inside another chain

In [ ]:
from operator import itemgetter

from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI

prompt1 = ChatPromptTemplate.from_template("what is the country {politician} is from?")
prompt2 = ChatPromptTemplate.from_template(
    "what continent is the country {country} in? respond in {language}"
)

# Original version: ChatOpenAI() used the library's default model,
# gpt-3.5-turbo, which OpenAI has since retired. We reuse MODEL.
model = ChatOpenAI(model=MODEL, temperature=1)

chain1 = prompt1 | model | StrOutputParser()

chain2 = (
    {"country": chain1, "language": itemgetter("language")}
    | prompt2
    | model
    | StrOutputParser()
)

chain2.invoke({"politician": "Miterrand", "language": "French"})

## Fallback for Chains
* When working with language models, you may often encounter issues from the underlying APIs, whether these be rate limiting or downtime. Therefore, as you go to move your LLM applications into production it becomes more and more important to safeguard against these. That's why LangChain introduced the concept of fallbacks.
* A fallback is an alternative plan that may be used in an emergency.
* Fallbacks can be applied not only on the LLM level but on the whole runnable level. This is important because often times different models require different prompts. So if your call to OpenAI fails, you don't just want to send the same prompt to Anthropic - you probably want to use a different prompt template and send a different version there.
* We can create fallbacks for LCEL chains. Here we do that with two different chains: ChatOpenAI (with a bad model name to easily create a chain that will error) and then a second chain with a different model and a plain-text prompt. Different models often work better with different prompts.

In [6]:
# First let's create a chain with a ChatModel
# We add in a string output parser here so the outputs between the two are the same type
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI

chat_prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "You're a funny assistant who always includes a joke in your response",
        ),
        ("human", "Who is the best {sport} player worldwide?"),
    ]
)
# Here we're going to use a bad model name to easily create a chain that will error
chat_model = ChatOpenAI(model="gpt-fake")

bad_chain = chat_prompt | chat_model | StrOutputParser()

#### Update note: the fallback chain
* The original fallback chain used OpenAI(), the completions model
  gpt-3.5-turbo-instruct. It was not a chat model, which is why the lesson
  uses a different, plain-text prompt for it. OpenAI has since retired that
  model, and current OpenAI models are all chat models.
* With the current models, we would do it like this: ChatOpenAI with MODEL.
  We keep the name llm and the plain-text prompt, so the example still shows
  the idea: the fallback chain can use a different prompt and model.
* A chat model returns a message, not plain text, so we add StrOutputParser
  to good_chain. That way both chains return the same type of output.
* The bad chain still uses the fake model name "gpt-fake" on purpose: it is
  the chain that fails, so that the fallback is used.

In [ ]:
# Now lets create a chain with the normal OpenAI model
from langchain_core.prompts import PromptTemplate
from langchain_openai import ChatOpenAI

prompt_template = """Instructions: You're a funny assistant who always includes a joke in your response.

Question: Who is the best {sport} player worldwide?"""

prompt = PromptTemplate.from_template(prompt_template)

# Original version: OpenAI() used the completions model
# gpt-3.5-turbo-instruct, which OpenAI has since retired. Current OpenAI
# models are chat models, so we use ChatOpenAI with MODEL. We keep the
# name llm so the rest of the example works the same way.
llm = ChatOpenAI(model=MODEL, temperature=1)

# A chat model returns a message, not plain text like the old completions
# model did. We add StrOutputParser so both chains return text, as the
# comment at the start of this example explains.
good_chain = prompt | llm | StrOutputParser()

In [ ]:
# We can now create a final chain which combines the two
chain = bad_chain.with_fallbacks([good_chain])

chain.invoke({"sport": "soccer"})

## How to execute the code from Visual Studio Code
* In Visual Studio Code, see the file 007-combining-chain.py
* In terminal, make sure you are in the directory of the file and run:
    * python 007-combining-chain.py